# Project: Feature Engineering for Machine Learning

Transform raw data into ML-ready features: encoding, scaling, binning, interaction features, and datetime features.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import (LabelEncoder, OneHotEncoder, StandardScaler,
                                   MinMaxScaler, RobustScaler, KBinsDiscretizer)
from sklearn.feature_selection import SelectKBest, f_regression

%matplotlib inline
print('Pandas version:', pd.__version__)

In [ ]:
np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'age': np.random.normal(38, 15, n).astype(int),
    'income': np.round(np.random.lognormal(10.8, 0.7, n), 2),
    'gender': np.random.choice(['M','F','Other'], n),
    'education': np.random.choice(['High School','Bachelor','Master','PhD'], n),
    'city': np.random.choice(['NYC','LA','Chicago','Houston','Phoenix','Seattle','Boston','Denver','Miami','Atlanta'], n),
    'signup_date': pd.date_range('2020-01-01', periods=n, freq='D'),
    'purchase_freq': np.random.poisson(5, n),
    'avg_order_value': np.round(np.random.lognormal(4, 0.5, n), 2),
})
df['target'] = (df['age']*0.1 + np.log(df['income'])*5 +
                df['purchase_freq']*2 + df['avg_order_value']*0.5 + np.random.normal(0, 10, n))
print('Raw shape:', df.shape)
df.head()

## 1. Encoding Categoricals

In [ ]:
edu_map = {'High School':0, 'Bachelor':1, 'Master':2, 'PhD':3}
df['education_ord'] = df['education'].map(edu_map)
df['gender_enc'] = LabelEncoder().fit_transform(df['gender'])

ohe = OneHotEncoder(sparse_output=False, drop='first')
city_enc = ohe.fit_transform(df[['city']])
city_cols = [f'city_{c}' for c in ohe.categories_[0][1:]]
city_df = pd.DataFrame(city_enc, columns=city_cols, index=df.index)
df = pd.concat([df, city_df], axis=1)
print(f'Added {len(city_cols)} one-hot columns')

## 2. Numerical Transformations

In [ ]:
df['income_log'] = np.log1p(df['income'])
df['spend_score'] = df['purchase_freq'] * df['avg_order_value']
print(f'Income skew: raw={df["income"].skew():.2f} -> log={df["income_log"].skew():.2f}')

scalers = {'Standard': StandardScaler(), 'MinMax': MinMaxScaler(), 'Robust': RobustScaler()}
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
num_feats = ['age','income','purchase_freq','avg_order_value','spend_score']
axes[0].boxplot(df[num_feats].values)
axes[0].set_title('Original')
axes[0].set_xticklabels(num_feats, rotation=45, fontsize=8)
for ax, (name, scaler) in zip(axes[1:], list(scalers.items())[:3]):
    ax.boxplot(scaler.fit_transform(df[num_feats]))
    ax.set_title(name)
    ax.set_xticklabels(num_feats, rotation=45, fontsize=8)
plt.tight_layout()
plt.show()

## 3. Feature Selection

In [ ]:
feature_cols = ['age','income_log','education_ord','gender_enc',
                'purchase_freq','avg_order_value','spend_score'] + city_cols
selector = SelectKBest(score_func=f_regression, k='all')
selector.fit(df[feature_cols], df['target'])
scores = pd.DataFrame({'feature': feature_cols, 'f_score': selector.scores_})\n    .sort_values('f_score', ascending=False)
plt.figure(figsize=(10, 6))
plt.barh(range(len(scores)), scores['f_score'], color='steelblue')
plt.yticks(range(len(scores)), scores['feature'])
plt.xlabel('F-Score')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Top 5 features:')
print(scores.head().to_string(index=False))

## Summary

- Ordinal, label, and one-hot encoding
- Log transform and interaction features
- Scaler comparison (Standard/MinMax/Robust)
- Feature selection with F-regression